# 会議の書き起こしから議事録を作る（決定事項・ToDo を JSON で抜き出す）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_summarization/meeting_minutes.ipynb)

NLP クックブック 26-10 ／ 要約と平易化

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

話者付きの会議の書き起こしから、決定事項・ToDo（担当者と期限）・論点を JSON で抜き出し、議事録にまとめたい。書き起こしにない担当者や期限を作っていないかも確かめたい。

In [1]:
import re
import unicodedata

# 架空の会議の書き起こし（このレシピのために作ったもの）。実際には文字起こしの結果を（話者, 発言）の列にして使う
transcript = [
    ("田中", "それでは、学習アプリ「まなぶん」のリリース準備会議を始めます。今日は、リリース日、対応する端末、料金、残っている作業の分担を決めたいと思います。"),
    ("佐藤", "開発の状況から報告します。主な機能はほぼできていますが、パスワードを再設定するとまれにログインできなくなる不具合が残っています。"),
    ("田中", "その不具合は、リリースまでに直せそうですか。"),
    ("佐藤", "原因の見当はついているので、11 月 28 日までには直せると思います。私が担当します。"),
    ("伊藤", "品質保証の立場から言うと、その修正のあとに回帰テストを 1 週間はやりたいです。"),
    ("田中", "そうすると、当初予定していた 12 月 3 日のリリースは厳しいですね。"),
    ("高橋", "営業としては、年内にはどうしても出したいです。12 月の第 2 週に、展示会で発表の場がありますので。"),
    ("佐藤", "修正が 28 日、テストが 1 週間だとすると、12 月 5 日ごろにテストが終わります。"),
    ("田中", "では、リリース日は 12 月 10 日にしましょう。展示会にも間に合います。皆さん、よろしいですか。"),
    ("鈴木", "異論ありません。"),
    ("伊藤", "それなら大丈夫です。"),
    ("田中", "リリース日は 12 月 10 日で決定とします。伊藤さん、テスト計画を作っていただけますか。"),
    ("伊藤", "はい、11 月 25 日までにテスト計画をまとめて共有します。"),
    ("田中", "次に、対応する端末です。iOS と Android を同時に出すかどうか。"),
    ("佐藤", "Android 版は、通知の機能がまだ不安定です。同時に出すと、テストの手間が倍になります。"),
    ("高橋", "お客様の半分くらいは Android を使っているので、できれば同時がいいのですが。"),
    ("伊藤", "正直なところ、両方を 1 週間でテストするのは無理です。"),
    ("田中", "初回のリリースは iOS 版だけにして、Android 版は来年 2 月を目標にしましょう。高橋さん、お客様への説明をお願いできますか。"),
    ("高橋", "わかりました。主なお客様 5 社には、私から直接説明します。"),
    ("田中", "続いて料金です。無料版と有料版を分ける方針は前回決めましたが、有料版の月額をいくらにするか。"),
    ("高橋", "競合が月額 800 円なので、500 円にして安さを打ち出したいです。"),
    ("鈴木", "ただ、500 円だと、サーバーの費用を考えると赤字にならないでしょうか。"),
    ("佐藤", "利用者が 1 万人を超えるまでは、500 円だと赤字になる計算です。"),
    ("田中", "料金は今日は決めきれませんね。経理の意見も聞いてから、次回の会議で決めます。高橋さん、競合の料金表をまとめておいてもらえますか。"),
    ("高橋", "はい、まとめておきます。"),
    ("田中", "次はデザインです。鈴木さん、アイコンはどうなっていますか。"),
    ("鈴木", "今の案は社内の評判があまりよくないので、作り直そうと思っています。来週の月曜までに、新しい案を 3 つ出します。"),
    ("田中", "お願いします。ストアに載せる紹介用の画像も必要ですね。"),
    ("鈴木", "画像はアイコンが決まってから作ります。"),
    ("田中", "それから、プレスリリースの文案も誰かが書かないといけません。"),
    ("高橋", "営業部でも書けますが、広報に頼んだほうがいいかもしれません。"),
    ("田中", "担当は広報と相談してから決めましょう。"),
    ("伊藤", "一つ気になっているのですが、利用者の学習記録を海外のサーバーに保存する点について、個人情報の扱いを法務に確認しましたか。"),
    ("佐藤", "まだです。利用規約の文面にも関わりますね。"),
    ("田中", "それは確認が必要です。佐藤さん、法務への確認をお願いします。リリースに関わるので、11 月 30 日までに結果を教えてください。"),
    ("佐藤", "承知しました。"),
    ("伊藤", "すみません、先ほどのテスト計画ですが、ログインの修正方針を見てから作りたいので、期限を 11 月 27 日に延ばしてもいいでしょうか。"),
    ("田中", "はい、27 日で構いません。"),
    ("田中", "最後に確認します。リリースは 12 月 10 日、初回は iOS 版のみ、料金は次回決定です。次回の会議は 12 月 1 日です。お疲れさまでした。"),
]
lines = [f"[{i}] {who}: {text}" for i, (who, text) in enumerate(transcript, 1)]   # 発言番号を付けて、根拠として指させるようにする
speakers = sorted({who for who, _ in transcript})

def make_chunks(lines, max_chars=600):
    """発言の途中で切らないように、発言単位で max_chars 文字ずつにまとめる"""
    chunks, cur = [], []
    for line in lines:
        if cur and sum(map(len, cur)) + len(line) > max_chars:
            chunks.append(cur); cur = []
        cur.append(line)
    return chunks + [cur]

chunks = make_chunks(lines)
print(sum(map(len, lines)), "文字 /", len(lines), "発言 / 参加者", speakers)
print(len(chunks), "分割:", [(c[0].split("]")[0] + "]", c[-1].split("]")[0] + "]", sum(map(len, c))) for c in chunks])

1856 文字 / 39 発言 / 参加者 ['伊藤', '佐藤', '田中', '鈴木', '高橋']
4 分割: [('[1]', '[12]', 587), ('[13]', '[23]', 532), ('[24]', '[36]', 555), ('[37]', '[39]', 182)]


In [2]:
import json
import sentencepiece as spm
import torch
from huggingface_hub import hf_hub_download
from pydantic import BaseModel, ValidationError, field_validator
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル（MIT ライセンス）。VRAM 約 7GB
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる（大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」を参照）。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
llm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto").eval()

def chat(messages, max_new_tokens=700):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

class Item(BaseModel):
    content: str
    evidence: list[int]          # 根拠の発言番号

class Todo(BaseModel):
    task: str
    owner: str | None            # 発言に担当者がなければ null
    due: str | None              # 発言に期限がなければ null
    evidence: list[int]

    @field_validator("owner", "due")
    @classmethod
    def none_if_unknown(cls, v):    # 「未定」「なし」などと文字で書いてきたものも null として扱う
        return None if v is None or re.match(r"\s*(未定|なし|特になし|不明|null)", v) else v

class Minutes(BaseModel):
    decisions: list[Item]
    todos: list[Todo]
    issues: list[Item]           # 結論が出ていない論点

INSTRUCTION = """次は会議の書き起こしの一部です。各行は「[発言番号] 話者: 発言」です。
決定事項 (decisions)、ToDo (todos)、結論が出ていない論点 (issues) を、例と同じ形の JSON で抜き出してください。
- 誰かが頼まれて引き受けたこと、自分からやると言ったこと、担当が決まっていないがやる必要があることは ToDo にする
- evidence には根拠となる発言番号を入れる
- ToDo の owner (担当者) と due (期限) は、発言の中で明示されたときだけ書く。明示されていなければ null にする。推測で補わない
- 該当するものがなければ空のリスト [] にする
- JSON 以外は出力しない"""
EXAMPLE_IN = """[1] 山田: 新しい名刺のデザインは A 案に決めます。
[2] 木村: 印刷会社への発注は私がやります。金曜までに済ませます。
[3] 山田: 配布の方法は、また今度考えましょう。社内報に載せる記事も、誰かに書いてもらわないと。"""
EXAMPLE_OUT = json.dumps({
    "decisions": [{"content": "名刺のデザインを A 案にする", "evidence": [1]}],
    "todos": [{"task": "印刷会社に発注する", "owner": "木村", "due": "金曜", "evidence": [2]},
              {"task": "社内報の記事を書く", "owner": None, "due": None, "evidence": [3]}],
    "issues": [{"content": "名刺の配布方法", "evidence": [3]}]}, ensure_ascii=False)

def extract_json(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.MULTILINE)
    obj, _ = json.JSONDecoder().raw_decode(text[text.index("{"):])
    return obj

def extract(chunk, max_retries=2):
    """（抜き出した結果, 出し直した回数）を返す。出し直しを使い切っても検証を通らなければ（None, 回数）を返す"""
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{EXAMPLE_IN}"}, {"role": "assistant", "content": EXAMPLE_OUT},
                {"role": "user", "content": f"{INSTRUCTION}\n\n" + "\n".join(chunk)}]
    for retry in range(max_retries + 1):
        out = chat(messages)
        try:
            return Minutes.model_validate(extract_json(out)), retry
        except (ValueError, ValidationError) as e:   # 壊れた JSON は、エラーを伝えて出し直させる
            messages += [{"role": "assistant", "content": out}, {"role": "user", "content": f"出力が不正です: {str(e)[:200]}\n例と同じ形の JSON だけを出力し直してください。"}]
    return None, max_retries

# 1. 分割ごとに抜き出す (map)。T4 なら、モデルのダウンロード（約 7GB）を除いて数分ほど
parts = []
for k, chunk in enumerate(chunks, 1):
    m, retry = extract(chunk)
    if m is None:    # 黙って空の結果にすると、その分割の決定事項や ToDo が消えたことに気づけない
        print(f"--- 分割 {k}: 出し直し {retry} 回でも有効な JSON が得られなかった。この分割は手で確かめる")
        continue
    parts.append(m)
    print(f"--- 分割 {k} (出し直し {retry} 回)")
    for t in m.todos:
        print("  ToDo:", t.model_dump())

--- 分割 1 (出し直し 1 回)
  ToDo: {'task': 'パスワード再設定不具合の修正', 'owner': '佐藤', 'due': '11 月 28 日', 'evidence': [4]}
  ToDo: {'task': '回帰テストの実施', 'owner': '伊藤', 'due': None, 'evidence': [5]}
  ToDo: {'task': 'テスト計画の作成', 'owner': '伊藤', 'due': None, 'evidence': [12]}
--- 分割 2 (出し直し 0 回)
  ToDo: {'task': 'テスト計画を 11 月 25 日までに共有する', 'owner': '伊藤', 'due': '11/25', 'evidence': [13]}
  ToDo: {'task': 'Android 版のテスト戦略を決定する', 'owner': None, 'due': None, 'evidence': [14, 15, 17]}
  ToDo: {'task': 'お客様への説明（高橋が担当）', 'owner': '高橋', 'due': None, 'evidence': [19]}
--- 分割 3 (出し直し 0 回)
  ToDo: {'task': '競合の料金表をまとめる', 'owner': '高橋', 'due': None, 'evidence': [25]}
  ToDo: {'task': '新しいアイコン案を 3 つ出す', 'owner': '鈴木', 'due': '来週の月曜', 'evidence': [27]}
--- 分割 4 (出し直し 0 回)
  ToDo: {'task': 'テスト計画の作成（ログイン修正方針後）', 'owner': '伊藤', 'due': '11/27', 'evidence': [37]}


In [3]:
import pandas as pd

# 2. 分割ごとの結果を 1 つにまとめる (reduce)。ToDo は根拠の発言番号の順に並べるだけで、LLM には渡さない
decisions = [d for m in parts for d in m.decisions]
issues = [d for m in parts for d in m.issues]
todos = sorted([t for m in parts for t in m.todos], key=lambda t: min(t.evidence or [0]))

# 3. 抜き出した担当者と期限が、根拠の発言に本当に書かれているかを機械的に確かめる
norm = lambda s: unicodedata.normalize("NFKC", s).replace(" ", "")

def check_todo(todo):
    problems = []
    ev = [transcript[i - 1] for i in todo.evidence if 1 <= i <= len(transcript)]
    if len(ev) < len(todo.evidence) or not ev:
        problems.append("発言番号が不正")
    ev_text = norm(" ".join(f"{who} {text}" for who, text in ev))
    if todo.owner is not None:
        owner = norm(todo.owner).removesuffix("さん")
        if owner not in speakers:
            problems.append("担当者が参加者にいない")
        elif owner not in ev_text:            # 根拠の発言の話者か本文に、担当者の名前が出てくるか
            problems.append("担当者が根拠の発言にない")
    if todo.due is not None:
        due = norm(todo.due)
        nums = re.findall(r"\d+", due)
        if due not in ev_text and not (nums and all(n in re.findall(r"\d+", ev_text) for n in nums)):
            problems.append("期限が根拠の発言にない")
    return problems

# 検査が働くことを、わざと作った誤り（書き起こしにない期限、参加者でない担当者）で確かめる
fakes = [Todo(task="プレスリリースの文案を書く", owner="高橋", due="12 月 1 日", evidence=[30, 31]),
         Todo(task="法務に確認する", owner="山本", due="11 月 30 日", evidence=[35])]
for t in fakes:
    print(t.task, t.owner, t.due, "→", check_todo(t))

rows = [{"ToDo": t.task, "担当": t.owner, "期限": t.due, "根拠": t.evidence, "検査": "、".join(check_todo(t)) or "OK"} for t in todos]
pd.set_option("display.max_colwidth", 40)
pd.DataFrame(rows)

プレスリリースの文案を書く 高橋 12 月 1 日 → ['期限が根拠の発言にない']
法務に確認する 山本 11 月 30 日 → ['担当者が参加者にいない']


,ToDo,担当,期限,根拠,検査
0,パスワード再設定不具合の修正,佐藤,11 月 28 日,[4],OK
1,回帰テストの実施,伊藤,None,[5],OK
2,テスト計画の作成,伊藤,None,[12],OK
3,テスト計画を 11 月 25 日までに共有する,伊藤,11/25,[13],OK
4,Android 版のテスト戦略を決定する,None,None,"[14, 15, 17]",OK
5,お客様への説明（高橋が担当）,高橋,None,[19],OK
6,競合の料金表をまとめる,高橋,None,[25],OK
7,新しいアイコン案を 3 つ出す,鈴木,来週の月曜,[27],OK
8,テスト計画の作成（ログイン修正方針後）,伊藤,11/27,[37],OK


In [4]:
# 4. 書き起こしを作った人（ここではレシピの作者）が付けた正解と比べる。task に含まれる語で対応を取る
gold = [("不具合|パスワード", "佐藤", "11月28日"), ("テスト計画", "伊藤", "11月27日"), ("説明", "高橋", None), ("料金表", "高橋", None),
        ("アイコン", "鈴木", "来週の月曜"), ("法務|個人情報", "佐藤", "11月30日"), ("プレスリリース", None, None)]
def same_due(a, b):
    """期限が同じか。どちらも None なら同じ。文字列が同じか、含まれる数字の並びが同じ（「11/27」と「11月27日」）なら同じとみなす"""
    if a is None or b is None:
        return a is None and b is None
    nums_a, nums_b = re.findall(r"\d+", norm(a)), re.findall(r"\d+", norm(b))
    return norm(a) == norm(b) or (nums_a != [] and nums_a == nums_b)

for key, owner, due in gold:
    found = [t for t in todos if re.search(key, t.task)]
    if not found:
        print(f"{key:<14} 見落とし")
    for t in found:
        print(f"{key:<14} 担当 {'○' if t.owner == owner else '×'} ({t.owner})  期限 {'○' if same_due(t.due, due) else '×'} ({t.due})  {t.task}")
print("正解にない ToDo:", [t.task for t in todos if not any(re.search(key, t.task) for key, _, _ in gold)])

不具合|パスワード      担当 ○ (佐藤)  期限 ○ (11 月 28 日)  パスワード再設定不具合の修正
テスト計画          担当 ○ (伊藤)  期限 × (None)  テスト計画の作成
テスト計画          担当 ○ (伊藤)  期限 × (11/25)  テスト計画を 11 月 25 日までに共有する
テスト計画          担当 ○ (伊藤)  期限 ○ (11/27)  テスト計画の作成（ログイン修正方針後）
説明             担当 ○ (高橋)  期限 ○ (None)  お客様への説明（高橋が担当）
料金表            担当 ○ (高橋)  期限 ○ (None)  競合の料金表をまとめる
アイコン           担当 ○ (鈴木)  期限 ○ (来週の月曜)  新しいアイコン案を 3 つ出す
法務|個人情報        見落とし
プレスリリース        見落とし
正解にない ToDo: ['回帰テストの実施', 'Android 版のテスト戦略を決定する']


In [5]:
# 5. 議事録の形に整え、全体の要約を付ける。要約の入力は書き起こしではなく、抜き出した JSON にする
def bullet(items):
    return "\n".join(f"- {d.content} (発言 {', '.join(map(str, d.evidence))})" for d in items) or "- なし"

digest = json.dumps({"decisions": [d.content for d in decisions], "todos": [f"{t.task} (担当: {t.owner or '未定'}、期限: {t.due or '未定'})" for t in todos],
                     "issues": [d.content for d in issues]}, ensure_ascii=False)
summary = chat([{"role": "user", "content": f"次は会議で決まったこと、ToDo、残った論点の一覧です。会議の内容を 3 文で要約してください。一覧にないことは書かないでください。\n\n{digest}"}], max_new_tokens=200)

def todo_line(t):
    line = f"- {t.task} / 担当: {t.owner or '未定'} / 期限: {t.due or '未定'}"
    problems = check_todo(t)
    if problems:     # 検査に引っかかったものには印を付けて、人が確かめるようにする
        line += f"  ※要確認 ({'、'.join(problems)})"
    return line

todo_lines = "\n".join(todo_line(t) for t in todos)
print(f"# 議事録: リリース準備会議\n参加者: {'、'.join(speakers)}\n\n## 要約\n{summary}\n\n## 決定事項\n{bullet(decisions)}\n\n## ToDo\n{todo_lines}\n\n## 論点 (未決)\n{bullet(issues)}")

# 議事録: リリース準備会議
参加者: 伊藤、佐藤、田中、鈴木、高橋

## 要約
会議では、学習アプリ「まなぶん」のリリース日を 12 月 10 日とし、まず iOS 版のみを初回リリースし、有料版の月額を 500 円とすることが決定された。Todo リストには、佐藤がパスワード再設定不具合の修正を担当し 11 月 28 日までに完了させるほか、伊藤が回帰テストやテスト計画の作成を担当し、期限はそれぞれ未定だが 11 月 25 日までにテスト計画をシェアすることが決まっている。また、高橋はお客様への説明や競合の料金表まとめ、鈴木は新しいアイコン案の提出を担当する。一方で、残った論点としては Android 版のテスト戦略、有料版の収益性、ストア用紹介画像、利用者の学習記録の個人情報の扱い、プレスリリースの文案、およびテスト計画の作成とログイン修正方針との関係が挙げられた。

## 決定事項
- 学習アプリ「まなぶん」のリリース日を 12 月 10 日とする (発言 9)
- iOS 版のみ初回リリース (発言 18)
- 有料版月額を 500 円とする (発言 21)

## ToDo
- パスワード再設定不具合の修正 / 担当: 佐藤 / 期限: 11 月 28 日
- 回帰テストの実施 / 担当: 伊藤 / 期限: 未定
- テスト計画の作成 / 担当: 伊藤 / 期限: 未定
- テスト計画を 11 月 25 日までに共有する / 担当: 伊藤 / 期限: 11/25
- Android 版のテスト戦略を決定する / 担当: 未定 / 期限: 未定
- お客様への説明（高橋が担当） / 担当: 高橋 / 期限: 未定
- 競合の料金表をまとめる / 担当: 高橋 / 期限: 未定
- 新しいアイコン案を 3 つ出す / 担当: 鈴木 / 期限: 来週の月曜
- テスト計画の作成（ログイン修正方針後） / 担当: 伊藤 / 期限: 11/27

## 論点 (未決)
- Android 版のテスト戦略 (発言 14, 15, 17)
- 有料版の収益性 (発言 22, 23)
- ストアに載せる紹介用画像 (発言 28)
- 利用者の学習記録の個人情報の扱い (発言 33)
- プレスリリースの文案 (発言 30)
- テスト計画の作成（ログイン修正方針との

- 書き起こしに発言番号を付け、抜き出した項目ごとに根拠の発言番号 (evidence) を答えさせる。人が確かめるときにすぐ原文に戻れるうえ、担当者や期限がその発言に本当に書かれているかを機械的に検査できる。JSON の取り出しと、壊れていたら出し直させる方法は、大規模言語モデルの章の「LLM の出力から JSON を取り出して検証する」と同じ（上の出力でも、分割 1 は 1 回目の出力が検証を通らず出し直している）。出し直しを使い切った分割は、空の結果にせずに知らせる。
- 書き起こしを発言の途中で切らないように分割し、分割ごとに抜き出してからまとめる（map-reduce。この章の「長い文書を分割して要約する」と同じ考え方）。議事録では、ある分割で決まった期限が後の分割で変わることがある。上の出力では、伊藤さんのテスト計画は [13] の 11 月 25 日が [37]〜[38] で 27 日に延びており、ToDo には期限なし [12]・11/25 [13]・11/27 [37] の 3 件が並んだ。期限なしの 1 件は、依頼 [12] が分割 1 の末尾に、引き受けと期限 [13] が分割 2 の先頭に分かれたために出た重複である（前の分割の末尾の数発言を重ねて入れると減らせる）。つなげただけでは重複や古い期限が残る（要約も 11 月 25 日と書いている）ので、同じ内容の ToDo は根拠の発言番号を見比べて人が見直す。
- 担当者と期限の検査は、担当者が参加者にいるか、担当者の名前と期限（の数字）が根拠の発言に出てくるかを確かめるだけの簡単なものである。上の出力では引っかかる ToDo はなく、わざと作った誤り（参加者でない「山本」、書き起こしにない「12 月 1 日」）は検出できた。一方、[31] の話者が高橋さんなので「プレスリリースの担当: 高橋」という誤りは通ってしまう。根拠の発言番号そのものの正しさ、「来週の月曜」のような相対的な日付、仕事の内容 (task) までは確かめられない。
- 検査は「書いたものに根拠があるか」しか見ないので、見落としや、決定でないものを決定とする誤りは見つからない。正解と比べると、7 件の ToDo のうち、法務への確認 ([35]) と担当未定のプレスリリースの文案 ([30]〜[32]) を ToDo として拾えなかった（どちらも論点としては出ているが、法務への確認は [35] で担当と期限まで決まっているので、ToDo から落ちたのは痛い）。逆に、誰も引き受けていない「Android 版のテスト戦略を決定する」を ToDo にしている（[18] で初回は iOS 版だけと決まっている。プロンプトの「担当が決まっていないがやる必要があることは ToDo にする」が、こうした項目を出しやすくしている可能性がある）。また、高橋さんの提案にすぎない [21] を「有料版の月額を 500 円とする」という決定事項にしている（実際は [24] で次回に持ち越し）。重要な会議では、決定事項は根拠の発言を人が読んで確かめるか、より大きなモデルを使う。
- 要約は書き起こし全体ではなく、抜き出した一覧から作っている。入力は短くて済むが、一覧の誤りはそのまま要約に入る（上の出力の要約も「有料版の月額を 500 円とすることが決定された」と書いている）。上の出力では「3 文で」と指示しても 4 文になっている。要約と原文の照合は、この章の「要約が原文と食い違っていないか検査する」を参照。
- 書き起こしはこのレシピのために作った架空の会議である。公開されている会議録には国立国会図書館の国会会議録検索システム（API あり）があるが、その利用条件によると、発言の著作権は原則として個々の発言者にあり、政治上の演説や陳述の利用、情報解析などの権利制限規定に当たる場合を除いて許諾が必要とされているので、使い方に応じて確かめる。社内の会議の書き起こしを外部の API に送るときは、社内の規程も確かめる。